In [1]:
import numpy as np
from pathlib import Path
import pandas as pd

saved_dir = Path(r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\saved_data\ROAD")

train_dfs = pd.read_pickle(saved_dir / "train_dfs.pkl")
test_dfs = pd.read_pickle(saved_dir / "test_dfs.pkl")

print("Reloaded train_dfs and test_dfs from saved_data/")
print("Train dfs:", len(train_dfs))
print("Test dfs:", len(test_dfs))
print("Train sizes:", [len(df) for df in train_dfs])
print("Test sizes:", [len(df) for df in test_dfs])

Reloaded train_dfs and test_dfs from saved_data/
Train dfs: 28
Test dfs: 28
Train sizes: [41580, 39911, 42657, 40945, 25197, 24186, 8058, 8024, 49098, 47143, 62576, 60064, 121688, 116802, 13400, 12862, 47249, 45352, 48501, 46554, 39720, 38128, 73526, 70575, 46872, 44992, 2393018, 850900]
Test sizes: [10394, 9977, 10664, 10236, 6299, 6046, 2014, 2006, 12274, 11785, 15644, 15016, 30421, 29200, 3349, 3215, 11812, 11338, 12125, 11638, 9930, 9531, 18381, 17643, 11718, 11247, 598254, 212724]


In [2]:
def prepare_windowed_dataset(
    train_dfs,
    test_dfs,
    data_col="Data",
    id_onehot_col="ID_onehot",
    attack_col="Attack",
    window_size=32,
    step=2,
):
    """
    Creates sliding-window datasets for training and testing.

    Returns
    -------
    windows : ndarray
        Training windows of shape (N, 54, 8)

    windows_test : ndarray
        Test windows of shape (M, 54, 8)

    attack_onehot : ndarray
        Training labels as one-hot vectors (R,T)

    attack_onehot_test : ndarray
        Test labels as one-hot vectors (R,T)
    """

    # ----------------------------------------------------
    # Convert CAN payload to uint8 array
    # ----------------------------------------------------
    def data_to_uint8_array(data):
        if isinstance(data, bytes):
            arr = np.frombuffer(data, dtype=np.uint8)

        elif isinstance(data, str):
            s = data.strip()

            if s.startswith(("0x", "0X")):
                s = s[2:]

            if any(ch in s for ch in (" ", "-", ":", ",")):
                parts = [
                    tok for tok in
                    s.replace("-", " ")
                     .replace(":", " ")
                     .replace(",", " ")
                     .split()
                    if tok
                ]

                if all(len(tok) == 2 for tok in parts):
                    arr = np.array([int(tok, 16) for tok in parts],
                                   dtype=np.uint8)
                else:
                    arr = np.frombuffer(
                        bytes.fromhex("".join(parts)),
                        dtype=np.uint8,
                    )

            else:
                arr = np.frombuffer(bytes.fromhex(s), dtype=np.uint8)

        elif isinstance(data, (list, tuple, np.ndarray, pd.Series)):
            arr = np.asarray(data, dtype=np.uint8).flatten()

        elif isinstance(data, int):
            arr = np.frombuffer(
                data.to_bytes(8, byteorder="big", signed=False),
                dtype=np.uint8,
            )

        else:
            raise TypeError(f"Unsupported Data type: {type(data)}")

        if arr.size < 8:
            arr = np.pad(arr, (0, 8 - arr.size))
        elif arr.size > 8:
            arr = arr[:8]

        return arr

    # ----------------------------------------------------
    # Process one dataset (train or test)
    # ----------------------------------------------------
    def process(dfs):

        payload_windows = []
        id_windows = []
        labels = []

        for df in dfs:

            payload = np.stack(
                [data_to_uint8_array(x) for x in df[data_col]]
            )

            ids = np.stack(df[id_onehot_col].tolist())

            if ids.shape[1:] != (22, 8):
                raise ValueError(
                    f"{id_onehot_col} must contain arrays of shape (22,8)"
                )

            attacks = df[attack_col].to_numpy()

            for start in range(0, len(df) - window_size, step):

                end = start + window_size

                # 32×8 payload window
                payload_window = payload[start:end]

                # summed 22×8 ID matrix
                id_sum = ids[start:end].sum(axis=0, dtype=np.uint16)

                # combine -> 54×8
                combined = np.concatenate(
                    [payload_window.astype(np.uint16), id_sum],
                    axis=0,
                )

                payload_windows.append(combined)

                # label window
                labels.append(
                    'T' if np.any(attacks[start:end] == 'T') else 'R'
                )

        payload_windows = np.asarray(payload_windows, dtype=np.uint16)
        labels = np.asarray(labels)

        # one-hot encoding
        attack_onehot = np.zeros((len(labels), 2), dtype=np.uint8)
        attack_onehot[labels == "R", 0] = 1
        attack_onehot[labels == "T", 1] = 1

        return payload_windows, attack_onehot

    # ----------------------------------------------------
    # Build datasets
    # ----------------------------------------------------
    windows, attack_onehot = process(train_dfs)
    windows_test, attack_onehot_test = process(test_dfs)

    return (
        windows,
        windows_test,
        attack_onehot,
        attack_onehot_test,
    )

windows, windows_test, attack_onehot, attack_onehot_test = prepare_windowed_dataset(
    train_dfs,
    test_dfs,
    window_size=32,
    step=2,
)

print(windows.shape)
print(windows_test.shape)
print(attack_onehot.shape)
print(attack_onehot_test.shape)

(2229345, 54, 8)
(556998, 54, 8)
(2229345, 2)
(556998, 2)


In [3]:
import torch
from torch.utils.data import Dataset


class CANDataset(Dataset):
    def __init__(
        self,
        windows,
        attack_onehot,
        train=True,
        is_spiking=False,
        time_window=100,
    ):
        # Convert to tensors
        self.windows = torch.as_tensor(windows, dtype=torch.float32)

        # Add channel dimension:
        # (N, H, W) -> (N, 1, H, W)
        if self.windows.ndim == 3:
            self.windows = self.windows.unsqueeze(1)

        attack_onehot = np.asarray(attack_onehot)

        # Convert one-hot -> class index
        if attack_onehot.ndim == 2:
            attack_labels = np.argmax(attack_onehot, axis=1)
        else:
            attack_labels = attack_onehot

        self.labels = torch.tensor(attack_labels, dtype=torch.long)

        self.is_spiking = is_spiking
        self.time_window = time_window

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):

        img = self.windows[index]
        label = self.labels[index]

        if self.is_spiking:
            # Optional normalization if inputs are not already in [0,1]
            #img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            # Generate spike train
            img = (
                torch.rand(self.time_window, *img.shape) < img
            ).float()

        return img, label

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

windows = scaler.fit_transform(
    windows.reshape(-1, windows.shape[-1])
).reshape(windows.shape)

windows_test = scaler.transform(
    windows_test.reshape(-1, windows_test.shape[-1])
).reshape(windows_test.shape)

from torch.utils.data import DataLoader

canbus_train = CANDataset(windows, attack_onehot, train=True, is_spiking=False)
train_loader = DataLoader(canbus_train, batch_size=2000, shuffle=True)

canbus_test = CANDataset(windows_test, attack_onehot_test, train=False, is_spiking=False)
test_loader = DataLoader(canbus_test, batch_size=2000, shuffle=False)

In [4]:
import torch.nn as nn
from tqdm.auto import tqdm

def create_ann(
    input_shape=(54, 8),
    num_classes=2,
    conv1_filters=32,
    conv2_filters=64,
    conv1_kernel_size=(3, 3),
    conv2_kernel_size=(3, 3),
):

    h, w = input_shape

    model = nn.Sequential(
        nn.Conv2d(
            1,
            conv1_filters,
            kernel_size=conv1_kernel_size,
        ),
        nn.ReLU(),

        nn.MaxPool2d((2,2)),

        nn.Conv2d(
            conv1_filters,
            conv2_filters,
            kernel_size=conv2_kernel_size,
        ),
        nn.ReLU(),

        nn.Flatten(),
    )

    # Automatically determine flatten size
    with torch.no_grad():
        dummy = torch.zeros(1, 1, h, w)
        flatten_size = model(dummy).shape[1]

    model.append(nn.Linear(flatten_size, 64))
    model.append(nn.ReLU())
    model.append(nn.Linear(64, num_classes))

    return model

import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import numpy as np


def train_ann(
    train_loader,
    test_loader,
    epochs=2,
    lr=1e-3,
    conv1_filters=32,
    conv2_filters=64,
    conv1_kernel_size=(3,3),
    conv2_kernel_size=(3,3),
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = create_ann(
        conv1_filters=conv1_filters,
        conv2_filters=conv2_filters,
        conv1_kernel_size=conv1_kernel_size,
        conv2_kernel_size=conv2_kernel_size,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ----------------------------
    # Training
    # ----------------------------

    model.train()

    
    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=False,
        )

        for x, y in pbar:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            output = model(x)

            loss = criterion(output, y)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            pred = output.argmax(dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)

            accuracy = correct / total

            pbar.set_postfix(
                loss=f"{running_loss/(pbar.n+1):.4f}",
                acc=f"{accuracy:.4f}",
            )

        print(
            f"Epoch {epoch+1}/{epochs}"
            f"  Loss={running_loss/len(train_loader):.4f}"
            f"  Acc={accuracy:.4f}"
        )

    # ----------------------------
    # Evaluation
    # ----------------------------

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)

            output = model(x)

            pred = torch.argmax(output, dim=1)

            y_true.extend(y.numpy())
            y_pred.extend(pred.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    report = classification_report(
        y_true,
        y_pred,
        target_names=["R", "T"],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tn = cm.sum() - (fp + fn + tp)

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    metrics = {
        "accuracy": round(accuracy, 4),
        "weighted_precision": round(precision, 4),
        "weighted_recall": round(recall, 4),
        "weighted_f1": round(f1, 4),
        "classification_report": report,
        "confusion_matrix": cm,
        "false_positive_rate": {
            "R": round(float(fpr[0]), 4),
            "T": round(float(fpr[1]), 4),
        },
        "false_negative_rate": {
            "R": round(float(fnr[0]), 4),
            "T": round(float(fnr[1]), 4),
        },
    }

    return model, metrics

In [7]:
import os
import gc
import time
from datetime import timedelta

import torch
import pandas as pd

RESULTS_FILE = "cnn_hyperparameter_search.csv"

# ==========================================================
# Load previous results
# ==========================================================

if os.path.exists(RESULTS_FILE):

    results_df = pd.read_csv(RESULTS_FILE)

    results = results_df.to_dict("records")

    completed = set(
        zip(
            results_df.conv1_filters,
            results_df.conv2_filters,
            results_df.conv1_kernel,
            results_df.conv2_kernel,
        )
    )

    print(f"Loaded {len(completed)} completed architectures.\n")

else:

    results = []
    completed = set()

# ==========================================================
# Hyperparameter search space
# ==========================================================

conv1_filters_list = [16, 32, 64]
conv2_filters_list = [16, 32, 64]

kernel_sizes = [
    (2, 2),
    (3, 3),
    (4, 4),
]

# ==========================================================
# Progress information
# ==========================================================

total_architectures = (
    len(conv1_filters_list)
    * len(conv2_filters_list)
    * len(kernel_sizes)
    * len(kernel_sizes)
)

completed_count = len(completed)

start_time = time.time()

print("=" * 70)
print(f"Total architectures : {total_architectures}")
print(f"Already completed   : {completed_count}")
print(f"Remaining           : {total_architectures - completed_count}")
print("=" * 70)

# ==========================================================
# Hyperparameter search
# ==========================================================

for conv1_filters in conv1_filters_list:
    for conv2_filters in conv2_filters_list:
        for conv1_kernel in kernel_sizes:
            for conv2_kernel in kernel_sizes:

                architecture = (
                    conv1_filters,
                    conv2_filters,
                    str(conv1_kernel),
                    str(conv2_kernel),
                )

                # Skip if already completed
                if architecture in completed:
                    print(f"Skipping {architecture}")
                    continue

                current = completed_count + 1

                percent = 100 * current / total_architectures

                elapsed = time.time() - start_time

                if completed_count > 0:
                    avg_time = elapsed / completed_count
                    eta = avg_time * (total_architectures - completed_count)
                else:
                    avg_time = 0
                    eta = 0

                print("\n" + "=" * 70)
                print(
                    f"Architecture {current}/{total_architectures} "
                    f"({percent:.1f}% complete)"
                )
                print("=" * 70)
                print(f"Conv1 filters : {conv1_filters}")
                print(f"Conv2 filters : {conv2_filters}")
                print(f"Conv1 kernel  : {conv1_kernel}")
                print(f"Conv2 kernel  : {conv2_kernel}")
                print()
                print(f"Elapsed time  : {timedelta(seconds=int(elapsed))}")
                print(f"ETA           : {timedelta(seconds=int(eta))}")
                print("=" * 70)

                architecture_start = time.time()

                try:

                    model, metrics = train_ann(
                        train_loader,
                        test_loader,
                        epochs=2,
                        conv1_filters=conv1_filters,
                        conv2_filters=conv2_filters,
                        conv1_kernel_size=conv1_kernel,
                        conv2_kernel_size=conv2_kernel,
                    )

                    architecture_time = time.time() - architecture_start

                    print("\nTraining complete")
                    print(f"Architecture time : {architecture_time:.1f} s")
                    print(f"Accuracy          : {metrics['accuracy']:.4f}")
                    print(f"Precision         : {metrics['weighted_precision']:.4f}")
                    print(f"Recall            : {metrics['weighted_recall']:.4f}")
                    print(f"Weighted F1       : {metrics['weighted_f1']:.4f}")

                    result = {
                        "conv1_filters": conv1_filters,
                        "conv2_filters": conv2_filters,
                        "conv1_kernel": str(conv1_kernel),
                        "conv2_kernel": str(conv2_kernel),
                        "accuracy": metrics["accuracy"],
                        "precision": metrics["weighted_precision"],
                        "recall": metrics["weighted_recall"],
                        "f1": metrics["weighted_f1"],
                        "status": "Success",
                        "error": "",
                    }

                except Exception as e:

                    architecture_time = time.time() - architecture_start

                    print("\nFAILED")
                    print(f"Architecture time : {architecture_time:.1f} s")
                    print(e)

                    result = {
                        "conv1_filters": conv1_filters,
                        "conv2_filters": conv2_filters,
                        "conv1_kernel": str(conv1_kernel),
                        "conv2_kernel": str(conv2_kernel),
                        "accuracy": None,
                        "precision": None,
                        "recall": None,
                        "f1": None,
                        "status": "Failed",
                        "error": str(e),
                    }

                # Save result immediately
                results.append(result)

                pd.DataFrame(results).to_csv(
                    RESULTS_FILE,
                    index=False,
                )

                completed.add(architecture)
                completed_count += 1

                print("Results saved.")

                # Cleanup
                try:
                    del model
                except Exception:
                    pass

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

# ==========================================================
# Final results
# ==========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "f1",
    ascending=False,
    na_position="last",
)

total_time = time.time() - start_time

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)
print(f"Total time : {timedelta(seconds=int(total_time))}")
print(f"Architectures tested : {len(results_df)}")
print("=" * 70)

print("\nTop 10 Architectures:\n")
print(results_df.head(10))

results_df.to_csv(RESULTS_FILE, index=False)

Loaded 64 completed architectures.

Total architectures : 81
Already completed   : 64
Remaining           : 17
Skipping (16, 16, '(2, 2)', '(2, 2)')
Skipping (16, 16, '(2, 2)', '(3, 3)')
Skipping (16, 16, '(2, 2)', '(4, 4)')
Skipping (16, 16, '(3, 3)', '(2, 2)')
Skipping (16, 16, '(3, 3)', '(3, 3)')
Skipping (16, 16, '(3, 3)', '(4, 4)')
Skipping (16, 16, '(4, 4)', '(2, 2)')
Skipping (16, 16, '(4, 4)', '(3, 3)')
Skipping (16, 16, '(4, 4)', '(4, 4)')
Skipping (16, 32, '(2, 2)', '(2, 2)')
Skipping (16, 32, '(2, 2)', '(3, 3)')
Skipping (16, 32, '(2, 2)', '(4, 4)')
Skipping (16, 32, '(3, 3)', '(2, 2)')
Skipping (16, 32, '(3, 3)', '(3, 3)')
Skipping (16, 32, '(3, 3)', '(4, 4)')
Skipping (16, 32, '(4, 4)', '(2, 2)')
Skipping (16, 32, '(4, 4)', '(3, 3)')
Skipping (16, 32, '(4, 4)', '(4, 4)')
Skipping (16, 64, '(2, 2)', '(2, 2)')
Skipping (16, 64, '(2, 2)', '(3, 3)')
Skipping (16, 64, '(2, 2)', '(4, 4)')
Skipping (16, 64, '(3, 3)', '(2, 2)')
Skipping (16, 64, '(3, 3)', '(3, 3)')
Skipping (16, 6

Epoch 1/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 1/2  Loss=0.3543  Acc=0.8454


Epoch 2/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 2/2  Loss=0.1578  Acc=0.9381

Training complete
Architecture time : 912.0 s
Accuracy          : 0.9014
Precision         : 0.9072
Recall            : 0.9014
Weighted F1       : 0.9032
Results saved.

Architecture 66/81 (81.5% complete)
Conv1 filters : 64
Conv2 filters : 32
Conv1 kernel  : (2, 2)
Conv2 kernel  : (4, 4)

Elapsed time  : 0:15:12
ETA           : 0:03:44

FAILED
Architecture time : 0.0 s
Calculated padded input size per channel: (26 x 3). Kernel size: (4 x 4). Kernel size can't be greater than actual input size
Results saved.

Architecture 67/81 (82.7% complete)
Conv1 filters : 64
Conv2 filters : 32
Conv1 kernel  : (3, 3)
Conv2 kernel  : (2, 2)

Elapsed time  : 0:15:12
ETA           : 0:03:27


...

Epoch 2/2  Loss=0.0947  Acc=0.9640

Training complete
Architecture time : 1002.1 s
Accuracy          : 0.9307
Precision         : 0.9303
Recall            : 0.9307
Weighted F1       : 0.9305
Results saved.

Architecture 78/81 (96.3% complete)
Conv1 filters : 64
Conv2 filters : 64
Conv1 kernel  : (3, 3)
Conv2 kernel  : (4, 4)

Elapsed time  : 2:03:08
ETA           : 0:06:23

FAILED
Architecture time : 0.0 s
Calculated padded input size per channel: (26 x 3). Kernel size: (4 x 4). Kernel size can't be greater than actual input size
Results saved.

Architecture 79/81 (97.5% complete)
Conv1 filters : 64
Conv2 filters : 64
Conv1 kernel  : (4, 4)
Conv2 kernel  : (2, 2)

Elapsed time  : 2:03:08
ETA           : 0:04:44


Epoch 1/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 1/2  Loss=0.2834  Acc=0.8778


Epoch 2/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 2/2  Loss=0.1090  Acc=0.9585

Training complete
Architecture time : 765.0 s
Accuracy          : 0.8946
Precision         : 0.8966
Recall            : 0.8946
Weighted F1       : 0.8954
Results saved.

Architecture 80/81 (98.8% complete)
Conv1 filters : 64
Conv2 filters : 64
Conv1 kernel  : (4, 4)
Conv2 kernel  : (3, 3)

Elapsed time  : 2:15:53
ETA           : 0:03:26

FAILED
Architecture time : 0.0 s
Calculated padded input size per channel: (25 x 2). Kernel size: (3 x 3). Kernel size can't be greater than actual input size
Results saved.

Architecture 81/81 (100.0% complete)
Conv1 filters : 64
Conv2 filters : 64
Conv1 kernel  : (4, 4)
Conv2 kernel  : (4, 4)

Elapsed time  : 2:15:53
ETA           : 0:01:41

FAILED
Architecture time : 0.0 s
Calculated padded input size per channel: (25 x 2). Kernel size: (4 x 4). Kernel size can't be greater than actual input size
Results saved.

SEARCH COMPLETE
Total time : 2:15:53
Architectures tested : 81

Top 10 Architectures:

    conv1_filter